# 📊 Anomaly Detection Model Evaluation & Benchmarks

Comprehensive model evaluation with metrics, benchmarks, and sensitivity analysis.

## Contents
1. Data Preparation
2. Statistical Methods Evaluation
3. ML Models Evaluation
4. Benchmark Comparison
5. Hyperparameter Sensitivity Analysis

In [ ]:
!pip install yfinance pandas numpy scikit-learn matplotlib seaborn tensorflow -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import tensorflow as tf
from tensorflow import keras
import yfinance as yf

plt.style.use('seaborn-v0_8-whitegrid')
print(f'TensorFlow: {tf.__version__}, GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# Fetch stock data
SYMBOLS = ['AAPL', 'GOOGL', 'MSFT', 'TSLA', 'NVDA']

def fetch_data(symbol, period='2y'):
    df = yf.Ticker(symbol).history(period=period).reset_index()
    df.columns = [c.lower() for c in df.columns]
    df['returns'] = df['close'].pct_change().fillna(0)
    return df

stock_data = {s: fetch_data(s) for s in SYMBOLS}
print(f'Loaded {len(stock_data)} stocks')

In [ ]:
def inject_anomalies(df, n=10, mag=0.15):
    """Inject synthetic anomalies for evaluation."""
    df_mod = df.copy()
    labels = np.zeros(len(df))
    np.random.seed(42)
    indices = np.random.choice(range(20, len(df)-20), min(n, len(df)//10), replace=False)
    for idx in indices:
        if np.random.random() > 0.5:
            df_mod.loc[idx, 'close'] *= (1 + mag)
        else:
            df_mod.loc[idx, 'close'] *= (1 - mag)
        labels[idx] = 1
    return df_mod, labels

eval_data = {s: inject_anomalies(stock_data[s]) for s in SYMBOLS}
print(f'Created evaluation datasets with injected anomalies')

In [ ]:
def compute_metrics(y_true, y_pred):
    if y_true.sum() == 0: return {'precision': 0, 'recall': 0, 'f1': 0}
    return {
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0)
    }

# Statistical detection
def detect_zscore(df, threshold=2.0):
    returns = df['returns']
    zscore = (returns - returns.mean()) / returns.std()
    return (np.abs(zscore) > threshold).astype(int).values

# Isolation Forest
def detect_isolation_forest(df, contamination=0.1):
    X = df[['close', 'volume', 'returns']].fillna(0).values
    X = StandardScaler().fit_transform(X)
    model = IsolationForest(contamination=contamination, random_state=42)
    return (model.fit_predict(X) == -1).astype(int)

In [ ]:
# Evaluate all methods
results = []

for symbol in SYMBOLS:
    df, labels = eval_data[symbol]
    
    # Z-Score
    pred = detect_zscore(df)
    m = compute_metrics(labels, pred)
    results.append({'Model': 'Z-Score', 'Symbol': symbol, **m})
    
    # Isolation Forest
    pred = detect_isolation_forest(df)
    m = compute_metrics(labels, pred)
    results.append({'Model': 'Isolation Forest', 'Symbol': symbol, **m})

results_df = pd.DataFrame(results)
print('\n📊 Results per symbol:')
print(results_df.round(3))

In [ ]:
# Aggregate benchmarks
benchmark = results_df.groupby('Model')[['precision', 'recall', 'f1']].mean()
print('\n📈 BENCHMARK COMPARISON:')
print(benchmark.round(3))

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
benchmark.plot(kind='bar', ax=ax)
ax.set_title('Model Performance Comparison', fontweight='bold')
ax.set_ylabel('Score')
ax.legend(loc='upper right')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('benchmarks.png', dpi=150)
plt.show()

In [ ]:
# Hyperparameter sensitivity
contaminations = [0.01, 0.05, 0.1, 0.15, 0.2]
sensitivity = []

df, labels = eval_data['AAPL']
for c in contaminations:
    pred = detect_isolation_forest(df, contamination=c)
    m = compute_metrics(labels, pred)
    sensitivity.append({'contamination': c, **m})

sens_df = pd.DataFrame(sensitivity)
print('\n🔬 Sensitivity Analysis:')
print(sens_df.round(3))

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sens_df['contamination'], sens_df['precision'], 'b-o', label='Precision')
ax.plot(sens_df['contamination'], sens_df['recall'], 'g-s', label='Recall')
ax.plot(sens_df['contamination'], sens_df['f1'], 'r-^', label='F1')
ax.set_xlabel('Contamination Rate')
ax.set_ylabel('Score')
ax.set_title('IF Sensitivity Analysis')
ax.legend()
plt.tight_layout()
plt.savefig('sensitivity.png', dpi=150)
plt.show()

In [ ]:
# Summary
print('\n' + '='*60)
print('📋 EVALUATION SUMMARY')
print('='*60)
print(f'Date: {datetime.now().strftime("%Y-%m-%d")}')
print(f'Stocks: {SYMBOLS}')
print('\nBest Model by F1-Score:')
best = benchmark['f1'].idxmax()
print(f'  {best}: F1={benchmark.loc[best, "f1"]:.3f}')
print('\n✅ Evaluation Complete')